In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  


In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# config
BRONZE_PATH  = "workspace.case_spark_cvm.bronze_cvm_informe_diario"
NOME_TABELA  = f"silver_cvm_informe_diario" 
SILVER_PATH = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

## CVM - Informações Diárias 

In [0]:
df_silver_cvm = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH, partition_col="data_processamento" )

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_silver_cvm = df_silver_cvm.withColumn(
    "CNPJ_FUNDO_CLASSE",
    PipelineConfig.normalizar_cnpj("CNPJ_FUNDO_CLASSE")
)

#### 1.1.2 Retirando dados duplicados

Com a mudança de rosolução da CVM (***Resolução CVM 175***), Com a nova regra, os fundos passaram a ser estruturados em classes e subclasses, adotando o tipo "CLASSES - FIF" (Fundo de Investimento Financeiro).
Caso acha dados do mesmo ***CNPJ_FUNDO_CLASSE***, os dados de "CLASSES - FIF" terão prioridade e o evento com nomecclatura antiga será excluido.


```
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
|TP_FUNDO_CLASSE| CNPJ_FUNDO_CLASSE|ID_SUBCLASSE| DT_COMPTC|   VL_TOTAL|      VL_QUOTA|VL_PATRIM_LIQ|CAPTC_DIA|RESG_DIA|NR_COTST|data_processamento|
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
|  CLASSES - FIF|12.586.174/0001-67|        NULL|2026-01-07|47445220.02|1.406455790000|  47448983.02|     0.00|    0.00|       1|          20260221|
|             FI|12.586.174/0001-67|        NULL|2026-01-07|47445414.65|1.406474270000|  47449606.58|     0.00|    0.00|       1|          20260221|
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
```

In [0]:
# Coluna temporaria para definir prioridade em CLASSES - FIF
df_silver_cvm = df_silver_cvm.withColumn(
    "prioridade_tipo",
    f.when(f.col("TP_FUNDO_CLASSE") ==  "CLASSES - FIF", 1).otherwise(2)
)

# Definindo a janela  particionando pelas colunas CORE
window_spec = Window.partitionBy("CNPJ_FUNDO_CLASSE", "ID_SUBCLASSE", "DT_COMPTC").orderBy("prioridade_tipo")

#Aplicamos a numeração das linhas (row_number) dentro de cada janela
df_silver_cvm = df_silver_cvm.withColumn("row_num", f.row_number().over(window_spec))

# filtrando prioridade_tipo = 1 de cada grupo e removendo as colunas auxiliares 
df_silver_cvm = df_silver_cvm.filter(f.col("row_num") == 1).drop("prioridade_tipo", "row_num")


In [0]:
# 1. Chaves que devem ser únicas
chave_negocio  = ["CNPJ_FUNDO_CLASSE", "ID_SUBCLASSE", "DT_COMPTC"]

# 2. Removemos as duplicatas 
# Como não á uma regra clara de desempate ficamos com a linha com maior patrimonio liquido
df_silver_cvm, df_quarentena_duplicadas = PipelineConfig.remover_duplicatas(
    df=df_silver_cvm,
    chave_negocio=chave_negocio,
    coluna_ordenacao="VL_PATRIM_LIQ" 
)

# 3. Salva a sujeira na quarentena
PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena_duplicadas, 
    tabela_origem="bronze_cvm_informe_diario", 
    data_proc=DATA_PROC
)


#### 1.1.3 Retirando dados nulos de Colunas Cores

In [0]:
regras_qualidade = {
    "CNPJ_FUNDO_CLASSE": "not_null",  # Não pode ser vazio (Substitui o dropna)
    "DT_COMPTC": "not_null",    # Não pode ser vazio (Substitui o dropna)
    "NR_COTST": "int",         # Não pode conter letras
    "VL_PATRIM_LIQ": "decimal"   # Não pode conter letras
}

df_silver_cvm, df_quarentena = PipelineConfig.aplicar_qualidade_e_separar(
    df=df_silver_cvm,
    regras=regras_qualidade
    )

PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena, 
    tabela_origem="bronze_cvm_informe_diario", 
    data_proc=DATA_PROC
)

#### 1.1.4 Tratamento do Tipo de Dado

In [0]:
# Dropando as colunas de metadados
df_silver_cvm = df_silver_cvm.drop("_source_url", "_ingest_timestamp", "data_processamento")


In [0]:
df_silver_cvm = df_silver_cvm.select(
    # Chaves de Identificação
    f.col('TP_FUNDO_CLASSE').cast(t.StringType()).alias('tp_fundo_classe'),
    f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType()).alias('cnpj_fundo_classe'),
    f.col('ID_SUBCLASSE').cast(t.StringType()).alias('id_subclasse'),
    
    # Data de Competência
    f.col('DT_COMPTC').cast(t.DateType()).alias('dt_comptc'),
    
    # Valores Financeiros e Patrimoniais (Alta Precisão)
    f.col('VL_TOTAL').cast(t.DecimalType(38, 2)).alias('vl_total'),
    f.col('VL_QUOTA').cast(t.DecimalType(38, 11)).alias('vl_quota'),
    f.col('VL_PATRIM_LIQ').cast(t.DecimalType(38, 2)).alias('vl_patrim_liq'),
    f.col('CAPTC_DIA').cast(t.DecimalType(38, 2)).alias('captc_dia'),
    f.col('RESG_DIA').cast(t.DecimalType(38, 2)).alias('resg_dia'),
    # Número de Cotistas
    f.col('NR_COTST').cast(t.LongType()).alias('nr_cotst')
)

### 1.2 Salvar na camada Silver

In [0]:
# Definindo as chaves estrangeiras 
chave_negocio = ["cnpj_fundo_classe", "id_subclasse", "dt_comptc"]

PipelineConfig.upsert_silver(
    spark=spark, 
    df_novo=df_silver_cvm, 
    tabela_destino= SILVER_PATH, 
    chave_negocio=chave_negocio
    )